# 03 · CNNs, RNNs & Sequence Models

| | |
|---|---|
| **Author** | Chandan C |
| **GitHub** | [github.com/chandanc5525](https://github.com/chandanc5525) |
| **Course** | Agentic AI BootCamp · Module 02 – Advanced PyTorch |
| **Repository** | [AgenticAI_BootCamp-/02_advanced_pytorch](https://github.com/chandanc5525/AgenticAI_BootCamp-/tree/main/02_advanced_pytorch) |
| **Level** | Intermediate |
| **Prerequisites** | Notebooks 01-02 |

---

### What you will learn
- Convolution, pooling, batch-norm, residual blocks
- Training a CNN with data augmentation
- Text pipeline: tokenisation → vocabulary → embeddings → padding
- RNN, LSTM, GRU, bidirectional models, packed sequences
- 1-D CNNs for text, pooling strategies
- Why sequence models led to attention

> Every cell runs on a plain CPU. If a GPU (CUDA / Apple MPS) is present, the notebooks use it automatically.

## 0. Setup

In [ ]:
import math, random, re, collections
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
import matplotlib.pyplot as plt

def seed_everything(s=42):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
seed_everything()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

# Part A - Convolutional Neural Networks
## 1. Convolution mechanics
`Conv2d(in_ch, out_ch, kernel, stride, padding)` slides learnable filters over the image.  
Output size: `floor((H + 2P - K) / S) + 1`.

In [ ]:
x = torch.randn(8, 3, 32, 32)                      # (batch, channels, H, W)
conv = nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1)
print("conv  :", conv(x).shape, "| params:", sum(p.numel() for p in conv.parameters()))
print("stride:", nn.Conv2d(3, 16, 3, stride=2, padding=1)(x).shape)
print("pool  :", nn.MaxPool2d(2)(conv(x)).shape)
print("adapt :", nn.AdaptiveAvgPool2d(1)(conv(x)).shape)   # resolution-independent

# a hand-made edge-detection filter
sobel = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]]).view(1, 1, 3, 3)
img = torch.zeros(1, 1, 8, 8); img[..., 4:] = 1
print(F.conv2d(img, sobel, padding=1)[0, 0, 3])

## 2. A synthetic image dataset
To stay offline we generate 16×16 images of **horizontal bars, vertical bars, and diagonals** with noise and random shifts. Swap in `torchvision.datasets.MNIST/CIFAR10` for real data - the model code stays identical.

In [ ]:
def make_image(cls, size=16):
    img = torch.zeros(size, size)
    pos = random.randint(3, size - 4); thick = random.randint(1, 2)
    if cls == 0:   img[pos:pos + thick, :] = 1                              # horizontal
    elif cls == 1: img[:, pos:pos + thick] = 1                              # vertical
    else:
        for i in range(size):
            j = min(size - 1, max(0, i + pos - size // 2)); img[i, j] = 1   # diagonal
    return img + 0.25 * torch.randn(size, size)

class ShapeDS(Dataset):
    def __init__(self, n=1500):
        self.y = torch.randint(0, 3, (n,))
        self.x = torch.stack([make_image(int(c)) for c in self.y]).unsqueeze(1)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.x[i], self.y[i]

ds = ShapeDS()
tr, va = random_split(ds, [1200, 300], generator=torch.Generator().manual_seed(0))
tr_dl, va_dl = DataLoader(tr, 64, shuffle=True), DataLoader(va, 256)
fig, ax = plt.subplots(1, 6, figsize=(10, 2))
for a, (im, lb) in zip(ax, [ds[i] for i in range(6)]): a.imshow(im[0], cmap="gray"); a.set_title(int(lb)); a.axis("off")
plt.show()

## 3. A CNN with batch-norm and a residual block

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.c1 = nn.Conv2d(ch, ch, 3, padding=1, bias=False); self.b1 = nn.BatchNorm2d(ch)
        self.c2 = nn.Conv2d(ch, ch, 3, padding=1, bias=False); self.b2 = nn.BatchNorm2d(ch)
    def forward(self, x):
        h = F.relu(self.b1(self.c1(x)))
        h = self.b2(self.c2(h))
        return F.relu(x + h)                       # skip connection: gradients flow freely

class SmallCNN(nn.Module):
    def __init__(self, n_classes=3):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(1, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU())
        self.block = ResBlock(16)
        self.down = nn.Sequential(nn.Conv2d(16, 32, 3, stride=2, padding=1), nn.BatchNorm2d(32), nn.ReLU())
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.2), nn.Linear(32, n_classes))
    def forward(self, x):
        return self.head(self.down(self.block(self.stem(x))))

def run_epoch(model, loader, opt=None):
    train = opt is not None
    model.train(train)
    tot, correct, n = 0., 0, 0
    with torch.set_grad_enabled(train):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb); loss = F.cross_entropy(out, yb)
            if train: opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item() * len(xb); correct += (out.argmax(1) == yb).sum().item(); n += len(xb)
    return tot / n, correct / n

cnn = SmallCNN().to(device)
opt = torch.optim.AdamW(cnn.parameters(), lr=3e-3, weight_decay=1e-3)
for ep in range(8):
    trl, tra = run_epoch(cnn, tr_dl, opt); vl, vacc = run_epoch(cnn, va_dl)
    print(f"ep {ep} | train {trl:.3f}/{tra:.3f} | val {vl:.3f}/{vacc:.3f}")

### 3.1 Data augmentation as tensor ops
`torchvision.transforms` does this at scale; the idea is just random tensor transformations.

In [ ]:
def augment(x):
    if random.random() < 0.5: x = torch.flip(x, dims=[-1])
    shift = random.randint(-2, 2); x = torch.roll(x, shifts=shift, dims=-2)
    return x + 0.05 * torch.randn_like(x)
print(augment(ds[0][0]).shape)
# try: import torchvision; torchvision.transforms.v2 for the full toolbox

### 3.2 Inspecting what the CNN learned

In [ ]:
w = cnn.stem[0].weight.detach().cpu()
fig, ax = plt.subplots(2, 8, figsize=(10, 3))
for a, f in zip(ax.ravel(), w): a.imshow(f[0], cmap="coolwarm"); a.axis("off")
plt.suptitle("First-layer filters"); plt.show()

# Part B - Text and Sequence Models
## 4. Text pipeline: tokenise → vocabulary → ids → embeddings
LLMs use sub-word tokenisers (BPE / SentencePiece). Here we keep a simple word-level tokeniser so the mechanics are visible.

In [ ]:
pos_words = "great excellent amazing good love wonderful fantastic nice happy brilliant".split()
neg_words = "bad terrible awful poor hate horrible boring sad worst disappointing".split()
neutral   = "the movie film plot actor story was is very really quite and but a an this that scene ending".split()
negators  = ["not", "never"]

def gen_sentence():
    label = random.randint(0, 1)
    n = random.randint(4, 14)
    words = [random.choice(neutral) for _ in range(n)]
    w = random.choice(pos_words if label else neg_words)
    if random.random() < 0.3:                       # negation flips meaning -> needs order awareness
        words.insert(random.randrange(n), "not " + w); label = 1 - label
    else:
        words.insert(random.randrange(n), w)
    return " ".join(words), label

data = [gen_sentence() for _ in range(4000)]
print(data[:3])

def tokenize(s): return re.findall(r"[a-z]+", s.lower())

counter = collections.Counter(t for s, _ in data for t in tokenize(s))
itos = ["<pad>", "<unk>"] + [w for w, c in counter.most_common()]
stoi = {w: i for i, w in enumerate(itos)}
encode = lambda s: torch.tensor([stoi.get(t, 1) for t in tokenize(s)])
print("vocab size:", len(itos), "| encode:", encode("this was not good"))

In [ ]:
class TextDS(Dataset):
    def __init__(self, data): self.data = data
    def __len__(self): return len(self.data)
    def __getitem__(self, i):
        s, y = self.data[i]; return encode(s), y

def collate(batch):
    seqs, ys = zip(*batch)
    lengths = torch.tensor([len(s) for s in seqs])
    return pad_sequence(seqs, batch_first=True, padding_value=0), lengths, torch.tensor(ys)

tds = TextDS(data)
ttr, tva = random_split(tds, [3400, 600], generator=torch.Generator().manual_seed(0))
ttr_dl = DataLoader(ttr, 64, shuffle=True, collate_fn=collate)
tva_dl = DataLoader(tva, 256, collate_fn=collate)
xb, lens, yb = next(iter(ttr_dl)); print(xb.shape, lens[:5], yb[:5])

## 5. RNN, LSTM, GRU
An RNN keeps a hidden state `h_t = f(x_t, h_{t-1})`. LSTM / GRU add **gates** to fight vanishing gradients. We use `pack_padded_sequence` so padding never influences the hidden state.

In [ ]:
class RNNClassifier(nn.Module):
    def __init__(self, vocab, emb=32, hid=64, kind="lstm", bidir=True, n_cls=2):
        super().__init__()
        self.emb = nn.Embedding(vocab, emb, padding_idx=0)
        rnn_cls = {"rnn": nn.RNN, "lstm": nn.LSTM, "gru": nn.GRU}[kind]
        self.rnn = rnn_cls(emb, hid, batch_first=True, bidirectional=bidir)
        self.fc = nn.Linear(hid * (2 if bidir else 1), n_cls)
        self.bidir, self.kind = bidir, kind
    def forward(self, x, lengths):
        e = self.emb(x)
        packed = pack_padded_sequence(e, lengths.cpu(), batch_first=True, enforce_sorted=False)
        out, h = self.rnn(packed)
        if self.kind == "lstm": h = h[0]                        # (h_n, c_n) -> h_n
        h = torch.cat([h[-2], h[-1]], 1) if self.bidir else h[-1]
        return self.fc(h)

def train_text(model, epochs=6, lr=3e-3):
    model.to(device); opt = torch.optim.AdamW(model.parameters(), lr=lr)
    for ep in range(epochs):
        model.train()
        for xb, lens, yb in ttr_dl:
            xb, yb = xb.to(device), yb.to(device)
            loss = F.cross_entropy(model(xb, lens), yb)
            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
    model.eval(); c = n = 0
    with torch.no_grad():
        for xb, lens, yb in tva_dl:
            c += (model(xb.to(device), lens).argmax(1).cpu() == yb).sum().item(); n += len(yb)
    return c / n

seed_everything()
for kind in ["rnn", "gru", "lstm"]:
    m = RNNClassifier(len(itos), kind=kind)
    print(f"{kind.upper():5s} val-acc = {train_text(m):.3f} | params = {sum(p.numel() for p in m.parameters()):,}")

## 6. A 1-D CNN and pooling strategies
Convolutions over time are fast and parallel; pooling collapses the sequence to a fixed vector. The same ideas (mean pooling with a mask) are used to build **sentence embeddings** later.

In [ ]:
def masked_mean(h, mask):                       # h: (B,T,C), mask: (B,T) bool
    m = mask.unsqueeze(-1).float()
    return (h * m).sum(1) / m.sum(1).clamp(min=1)

class CNNText(nn.Module):
    def __init__(self, vocab, emb=32, ch=64, n_cls=2):
        super().__init__()
        self.emb = nn.Embedding(vocab, emb, padding_idx=0)
        self.convs = nn.ModuleList([nn.Conv1d(emb, ch, k, padding=k // 2) for k in (2, 3, 5)])
        self.fc = nn.Linear(ch * 3, n_cls)
    def forward(self, x, lengths=None):
        e = self.emb(x).transpose(1, 2)                             # (B, emb, T)
        feats = [F.relu(c(e)).amax(dim=2) for c in self.convs]       # global max-pool
        return self.fc(torch.cat(feats, 1))

seed_everything()
print("1D-CNN val-acc:", round(train_text(CNNText(len(itos))), 3))
h = torch.randn(2, 5, 4); mask = torch.tensor([[1,1,1,0,0],[1,1,1,1,1]]).bool()
print("masked mean:", masked_mean(h, mask).shape)

## 7. Looking at embeddings
Words used in similar contexts should land close together.

In [ ]:
model = RNNClassifier(len(itos), kind="gru"); train_text(model, epochs=6)
E = F.normalize(model.emb.weight.detach().cpu(), dim=-1)
def neighbours(word, k=5):
    sims = E @ E[stoi[word]]
    return [itos[i] for i in sims.topk(k + 1).indices[1:]]
print("good ->", neighbours("good"))
print("bad  ->", neighbours("bad"))

## 8. Limitations → why Transformers
| Problem with RNNs | Consequence |
|---|---|
| Sequential computation | Cannot parallelise over time → slow training |
| Information squeezed into one hidden vector | Long-range dependencies are forgotten |
| Fixed processing path | Hard to focus on relevant tokens |

**Attention** lets every token look directly at every other token - the topic of the next notebook.

In [ ]:
# Quick experiment: gradient flow vs. sequence length in a vanilla RNN
def grad_to_first_token(T):
    rnn = nn.RNN(8, 8, batch_first=True); x = torch.randn(1, T, 8, requires_grad=True)
    out, _ = rnn(x); out[:, -1].sum().backward()
    return x.grad[0, 0].norm().item()
for T in [5, 20, 50, 100]: print(f"T={T:3d} | grad norm at first token = {grad_to_first_token(T):.2e}")

## Exercises
1. Add data augmentation to the CNN training loop and compare validation accuracy.
2. Replace `nn.Embedding` with pre-trained vectors (GloVe) and freeze / fine-tune them.
3. Add an attention-pooling layer on top of the GRU outputs instead of using the last hidden state.
4. Turn the GRU into a *language model* (predict the next word) and sample sentences.
5. Compare `padding_idx`, packed vs. unpacked sequences: measure how accuracy changes.

---
### Summary & next step
Work through the exercises above before moving on. Next notebook: **04_attention_and_transformers_from_scratch.ipynb**.

*Notebook by **Chandan C** · [github.com/chandanc5525](https://github.com/chandanc5525) · Agentic AI BootCamp*